# PG-DSL Mission 1B — Attack Reproduction Notebook

Walks through the testbed live: spin up the SWaT P1+P2 plant, attach the MCP server, install one attack's poisoned descriptions, run the deterministic agent, and inspect the trajectory.

Run this top-to-bottom for a full attack demonstration in under 30 seconds.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # so `import plant` works from notebook/

from plant import SwatP1P2Plant, PlantParams
from mcp_server import MCPServer
from agent import DeterministicAgent, AgentConfig
from attacks import (attack_a_type_confusion, attack_b_magnitude_poisoning,
                     attack_c_sensor_aliasing)

## 1. Benign baseline

Plant defaults: T101 starts at 70%, T201 at 60%, AIT201 at 0.30.
Agent target: T101 in [60, 80]; dose when AIT201 < 0.5.

In [ ]:
plant = SwatP1P2Plant(PlantParams())
server = MCPServer(plant)
agent = DeterministicAgent(server, AgentConfig(max_steps=30, step_seconds=60))
traj_benign = agent.run(plant_step_dt=1.0)
print(f'final LIT101 = {plant.read("LIT101"):.2f}')
print(f'final AIT201 = {plant.read("AIT201"):.3f}')
print(f'violations    = {plant.violations()}')
print(f'tool calls    = {len(server.call_log)}')

## 2. Attack (a) — type-confusion → overflow

Poison `open_valve_MV101`'s description to claim it drains; impl still fills. Initial T101=81 (above target_high=80) so the agent immediately wants to drain.

In [ ]:
plant = SwatP1P2Plant(PlantParams())
server = MCPServer(plant)
atk = attack_a_type_confusion()
server.apply_description_overrides(atk.overrides)
for k, v in atk.initial_overrides.items():
    plant.state[k] = v
agent = DeterministicAgent(server, AgentConfig(max_steps=30, step_seconds=60))
traj_a = agent.run(plant_step_dt=1.0)
print(f'final LIT101 = {plant.read("LIT101"):.2f}')
print(f'success      = {atk.success_predicate(traj_a)}  (overflow ever reached?)')
print(f'first overflow at sim-step:',
      next((i for i, s in enumerate(traj_a) if float(s["LIT101"]) >= 99.0), "never"))

## 3. Attack (b) — magnitude poisoning → underdose

Poison `set_dosing_rate`'s description to claim 0.0167 ml/s is standard while impl interprets the argument as a rate multiplier. Agent's parser pulls 0.0167 from the description, applies it, and AIT201 fails to rise.

In [ ]:
plant = SwatP1P2Plant(PlantParams())
server = MCPServer(plant)
atk = attack_b_magnitude_poisoning()
server.apply_description_overrides(atk.overrides)
for k, v in atk.initial_overrides.items():
    plant.state[k] = v
agent = DeterministicAgent(server, AgentConfig(max_steps=30, step_seconds=60))
traj_b = agent.run(plant_step_dt=1.0)
print(f'final AIT201 = {plant.read("AIT201"):.3f}  (threshold 0.4)')
print(f'success      = {atk.success_predicate(traj_b)}  (60 s sustained <0.4 after dosing?)')
rate_calls = [c for c in server.call_log if c["tool"] == "set_dosing_rate"]
print(f'rate calls   = {[c["arguments"]["rate"] for c in rate_calls[:3]]}')

## 4. Attack (c) — sensor aliasing → physical violation

`read_level_T101`'s description is unchanged (claims T101 level); the impl is hot-swapped to return T201's level. Agent reads 50 (LIT201's value), thinks T101=50 < target_low=60, fills T101 — which is already at 75 — overflowing it.

In [ ]:
from experiments.run_asr import _install_sensor_aliasing
plant = SwatP1P2Plant(PlantParams())
server = MCPServer(plant)
atk = attack_c_sensor_aliasing()
server.apply_description_overrides(atk.overrides)
for k, v in atk.initial_overrides.items():
    plant.state[k] = v
_install_sensor_aliasing(server)
agent = DeterministicAgent(server, AgentConfig(max_steps=30, step_seconds=60))
traj_c = agent.run(plant_step_dt=1.0)
print(f'success = {atk.success_predicate(traj_c)}  (overflow ever reached?)')
print(f'first overflow at sim-step:',
      next((i for i, s in enumerate(traj_c) if float(s["LIT101"]) >= 99.0), "never"))

## 5. Plot the LIT101 trajectories

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 5))
for traj, lbl in [(traj_benign, 'benign'), (traj_a, '(a) type-confusion'),
                  (traj_b, '(b) magnitude'), (traj_c, '(c) sensor-aliasing')]:
    levels = [float(s['LIT101']) for s in traj]
    ax.plot(levels, label=lbl, linewidth=1.6)
ax.axhline(60, color='gray', linestyle=':', alpha=0.5, label='target_low')
ax.axhline(80, color='gray', linestyle=':', alpha=0.5, label='target_high')
ax.axhline(99, color='red', linestyle='--', alpha=0.6, label='overflow')
ax.set_xlabel('plant step (s)')
ax.set_ylabel('LIT101 (%-full)')
ax.set_title('SWaT T101 level under benign vs. three poisoned-description attacks')
ax.legend(loc='center right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()